# Minería de Texto · Unidad I

## Práctica guiada en Python

En esta práctica voy a limpiar tres comentarios, separar las palabras, contar el vocabulario, quitar stopwords y revisar las frecuencias. También probaré la limpieza con un texto que contiene HTML.

## 1. Preparar el entorno y el corpus

In [ ]:
import re
from collections import Counter

corpus = [
    "¡El servicio al cliente es excelente y muy rápido en la entrega!",
    "El servicio es pésimo y la atención al cliente es muy lenta.",
    "Excelente atención pero el servicio de entrega es lento.",
]

print("Librerías listas")
for i, d in enumerate(corpus, start=1):
    print(f"D{i}: {d}")

## 2. Limpiar HTML y ruido

Primero quito etiquetas HTML, enlaces, correos y emojis. Después normalizo los espacios.

### Quitar etiquetas HTML

In [ ]:
sucio = "<p>¡El <b>servicio</b> al cliente es <i>excelente</i>!</p>"
limpio_html = re.sub(r"<[^>]+>", "", sucio)
print(sucio)
print(limpio_html)

### Quitar URLs, correos, emojis y espacios repetidos

In [ ]:
ruido = "Visítanos en https://ejemplo.com o escríbenos a info@ejemplo.com ☺ ¡Gracias!"
ruido = re.sub(r"https?://\S+", "", ruido)
ruido = re.sub(r"\S+@\S+", "", ruido)
ruido = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", "", ruido)
ruido = re.sub(r"\s+", " ", ruido).strip()
print(ruido)

### Reunir la limpieza en una función

In [ ]:
def limpiar_html_y_ruido(texto):
    texto = re.sub(r"<[^>]+>", "", texto)  # etiquetas HTML
    texto = re.sub(r"https?://\S+", "", texto)  # URLs
    texto = re.sub(r"\S+@\S+", "", texto)  # correos
    texto = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", "", texto)  # emojis
    texto = re.sub(r"\s+", " ", texto).strip()  # espacios repetidos
    return texto

### Aplicar la limpieza al corpus

In [ ]:
corpus = [limpiar_html_y_ruido(d) for d in corpus]
for i, d in enumerate(corpus, start=1):
    print(f"D{i}: {d}")

## 3. Normalizar: minúsculas y puntuación

### Convertir a minúsculas y quitar puntuación

In [ ]:
d1 = corpus[0].lower()
limpio = re.sub(r"[¡!¿?.,;:]", "", d1)
print(limpio)

### Crear la función de normalización y aplicarla al corpus

In [ ]:
def normalizar(texto):
    texto = limpiar_html_y_ruido(texto)  # HTML, URLs, correos y emojis
    texto = texto.lower()
    texto = re.sub(r"[¡!¿?.,;:]", "", texto)
    return texto

docs_norm = [normalizar(d) for d in corpus]
for i, d in enumerate(docs_norm, start=1):
    print(f"d{i} = {d}")

assert "pésimo" in docs_norm[1]  # los acentos se conservan

In [ ]:
def normalizar(texto):
    texto = limpiar_html_y_ruido(texto)
    texto = texto.lower()
    texto = re.sub(r"[¡!¿?.,;:]", "", texto)
    return texto

docs_norm = [normalizar(d) for d in corpus]
for i, d in enumerate(docs_norm, start=1):
    print(f"d{i} = {d}")

assert "pésimo" in docs_norm[1]  # los acentos se conservan

## 4. Tokenizar y contar los tokens

Con `split()` separo cada comentario en palabras.

In [ ]:
tokens = [d.split() for d in docs_norm]
for i, t in enumerate(tokens, start=1):
    print(f"T{i} = {t}")

n = [len(t) for t in tokens]
N = sum(n)
print("n =", n, " N =", N)

## 5. Construir el vocabulario

El vocabulario contiene las palabras diferentes del corpus. Uso conjuntos para quitar repetidas.

In [ ]:
print(len(tokens[1]), len(set(tokens[1])))  # lista vs. conjunto

V = set(tokens[0])
for t in tokens[1:]:
    V = V | set(t)

V = sorted(V)
print(V)
print("|V| =", len(V))
print("Repetidos descartados:", N - len(V))

## 6. Quitar stopwords

Uso el conjunto de palabras indicado en la práctica.

In [ ]:
S = {"al", "de", "el", "en", "es", "la", "muy", "pero", "y"}
print("¿S está dentro de V?", S <= set(V))

V_filtrado = sorted(set(V) - S)
print("Vocabulario filtrado:", V_filtrado)
print("|S| =", len(S), " |V_filtrado| =", len(V_filtrado))
print("% stopwords =", len(S) / len(V) * 100)

## 7. Contar frecuencias (TF)

Cuento los términos filtrados en cada comentario y después sumo sus frecuencias.

In [ ]:
vf = set(V_filtrado)
tf = [Counter(w for w in t if w in vf) for t in tokens]
for i, c in enumerate(tf, start=1):
    print(f"TF(D{i}) =", dict(c))

tf_total = sum(tf, Counter())
print("Frecuencia total:", tf_total.most_common())

## 8. Ficha de métricas y comprobaciones

In [ ]:
ficha = {
    "N (tokens totales)": N,
    "|V| (vocabulario)": len(V),
    "|S| (stopwords)": len(S),
    "|V_filtrado|": len(V_filtrado),
}
for nombre, valor in ficha.items():
    print(f"{nombre:<20} {valor}")

assert N == 33
assert len(V) == 18
assert len(S) == 9
assert len(S) / len(V) == 0.5
assert len(V_filtrado) == len(V) - len(S)
print("Todas las verificaciones pasaron")

## 9. Gráfica de frecuencias

In [ ]:
comentario_extra = "<p>La <b>atención</b> fue buena y la entrega rápida.</p>"
corpus_extra = corpus + [comentario_extra]
docs_extra = [normalizar(d) for d in corpus_extra]
tokens_extra = [d.split() for d in docs_extra]

N_extra = sum(len(t) for t in tokens_extra)
V_extra = set()
for t in tokens_extra:
    V_extra = V_extra | set(t)
V_filtrado_extra = sorted(V_extra - S)

print("Comentario extra normalizado:", docs_extra[3])
print("N original =", N, "| N con el cuarto comentario =", N_extra)
print("|V| original =", len(V), "| |V| con el cuarto comentario =", len(V_extra))
print("|V_filtrado| original =", len(V_filtrado), "| |V_filtrado| con el cuarto comentario =", len(V_filtrado_extra))

## 10. Retos opcionales

In [ ]:
import nltk
nltk.download("stopwords")

In [ ]:
from nltk.corpus import stopwords

stopwords_es = set(stopwords.words("spanish"))
S_nltk = set(V) & stopwords_es
V_filtrado_nltk = sorted(set(V) - stopwords_es)

print("Vocabulario filtrado con las stopwords de la práctica:", V_filtrado)
print("Stopwords de NLTK presentes en V:", sorted(S_nltk))
print("Vocabulario filtrado con NLTK:", V_filtrado_nltk)
print("|V_filtrado| con la práctica =", len(V_filtrado))
print("|V_filtrado| con NLTK =", len(V_filtrado_nltk))

In [ ]:
comentario_extra = "<p>La <b>atención</b> fue buena y la entrega rápida.</p>"

# Agrego el comentario a una copia para conservar las métricas principales de la práctica.
corpus_extra = corpus + [comentario_extra]
docs_extra = [normalizar(d) for d in corpus_extra]
tokens_extra = [d.split() for d in docs_extra]
N_extra = sum(len(t) for t in tokens_extra)

V_extra = set()
for t in tokens_extra:
    V_extra = V_extra | set(t)
V_extra = sorted(V_extra)
V_filtrado_extra = sorted(set(V_extra) - S)

print("Comentario extra normalizado:", docs_extra[3])
print("Tokens del comentario extra:", tokens_extra[3])
print("N original =", N, "| N con el cuarto comentario =", N_extra)
print("|V| original =", len(V), "| |V| con el cuarto comentario =", len(V_extra))
print("|V_filtrado| original =", len(V_filtrado), "| |V_filtrado| con el cuarto comentario =", len(V_filtrado_extra))

### Reto 2. Comparar con stopwords de NLTK

Esta parte descarga el recurso de stopwords en español. En Colab se puede ejecutar directamente.

In [ ]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stopwords_es = set(stopwords.words("spanish"))
S_nltk = set(V) & stopwords_es
V_filtrado_nltk = sorted(set(V) - stopwords_es)

print("Vocabulario filtrado con las stopwords de la práctica:", V_filtrado)
print("Stopwords de NLTK presentes en V:", sorted(S_nltk))
print("Vocabulario filtrado con NLTK:", V_filtrado_nltk)
print("|V_filtrado| con la práctica =", len(V_filtrado))
print("|V_filtrado| con NLTK =", len(V_filtrado_nltk))